# 09 · Nonlinear geometry — worked solutions

These solutions are most useful after attempting the chapter exercises. Each
section states the reasoning before the calculation; numerical values depend on
the compact, fixed-seed setup below.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import geodef

rng = np.random.default_rng(0)
fault = geodef.Fault.planar(
    lat=34.0, lon=-118.0, depth=9_000.0, strike=90.0, dip=30.0,
    length=24_000.0, width=12_000.0, n_length=4, n_width=3,
)
i = np.arange(fault.n_patches) % 4
j = np.arange(fault.n_patches) // 4
true_slip = np.exp(-((i - 1.5) / 1.2) ** 2 - ((j - 1.0) / 0.9) ** 2)
station_lon, station_lat = np.meshgrid(
    np.linspace(-118.18, -117.82, 6), np.linspace(33.90, 34.12, 5)
)
station_lon, station_lat = station_lon.ravel(), station_lat.ravel()
east, north, up = fault.displacement(
    station_lat, station_lon, slip_strike=0.0, slip_dip=true_slip
)
sigma = 0.004
gnss = geodef.data.gnss(
    lon=station_lon, lat=station_lat,
    east=east + rng.normal(0.0, sigma, east.size),
    north=north + rng.normal(0.0, sigma, north.size),
    up=up + rng.normal(0.0, sigma, up.size),
    sigma_east=sigma, sigma_north=sigma, sigma_up=sigma,
    name="solution_gnss",
)


## 1–2. Scan and restart

Use one regularization choice for every trial. The scan reveals the basin;
refinement from both sides tests local optimizer dependence.

## 3. Noise

Higher noise flattens the projected objective and broadens acceptable geometry.

## 4. Depth–width trade-off

A deeper, wider source can reproduce similar broad deformation by changing
slip amplitude. Plot a surface, not only marginal scans.


In [ ]:
def projected_misfit(dip):
    trial = geodef.Fault.planar(
        lat=34.0, lon=-118.0, depth=9_000.0, strike=90.0, dip=float(dip),
        length=24_000.0, width=12_000.0, n_length=4, n_width=3,
    )
    fit = geodef.solve(
        trial, datasets=gnss, components="dip", regularization="laplacian",
        regularization_strength=1.0,
    )
    return float(np.sum((fit.residuals / gnss.sigma) ** 2))

dips = np.arange(15.0, 56.0, 5.0)
misfits = np.array([projected_misfit(dip) for dip in dips])
print("best scanned dip:", dips[np.argmin(misfits)])
print("misfit curve:", misfits)


## Interpretation and alternatives

A broad minimum is an identifiability result, not an optimizer failure.
